# Experiment 3A: Hidden Per-Spike Jitter — SHD (eval-only)

## Overview

This notebook is the **eval-only** companion to `jitter_train.ipynb`.
It loads pretrained models from `data` (the checkpoints
produced by `jitter_train.ipynb` with `eval_all_variation = True`) and sweeps
hidden-layer per-spike Gaussian jitter at evaluation time. **No training is performed.**

Since jitter / shift / deletion all train identical networks under the
train-clean / eval-perturbed protocol (same architecture, seed, data,
loss, optimiser), the jitter checkpoints serve as a single "clean-trained
SHD model" cache that all three Phase 3 experiments can reuse.

**Architecture:** Input → 128 hidden → 128 hidden → 20 output (SRMALPHA)

**Sweep (eval only):** sigma in {0, 1, 3, 5, 10, 17, 25} ms

A global `eval_all_variation` flag selects whether to load+sweep one
`(DATASET_KEY, USE_DELAY)` configuration or all six.

## 1. Imports and Setup

In [ ]:
import os
import sys
import json
import random
from typing import Optional

import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Add SLAYER to path
CURRENT_DIR = os.getcwd()
sys.path.append(os.path.join(CURRENT_DIR, "../../../temporal_shd_project/code/src"))
import slayerSNN as snn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Global Configuration

All key hyper-parameters and switches are defined here. `MODEL_DIR` points at
the directory holding the jitter checkpoints to load.

In [ ]:
# =====================================================================
# When True: load and sweep every combination of DATASET_KEY in
# ("whole", "part", "norm") × USE_DELAY in (False, True) — six cells of
# the grid. The single (DATASET_KEY, USE_DELAY) below is used as the
# default config when this flag is False.
# =====================================================================
eval_all_variation: bool = False

# =====================================================================
# Network variant: set to True for SGD-delay, False for SGD (no delay)
# =====================================================================
USE_DELAY: bool = False

# =====================================================================
# Dataset variant: "whole", "part", or "norm"
# =====================================================================
DATASET_KEY: str = "whole"

# --- Dataset configurations ---
DATASET_CONFIGS = {
    "whole": {"mat_file": "../../realistic/shd/shd_data/shd_whole.mat", "input_dim": 700},
    "part":  {"mat_file": "../../realistic/shd/shd_data/shd_part_new.mat", "input_dim": 224},
    "norm":  {"mat_file": "../../realistic/shd/shd_data/shd_norm_new.mat", "input_dim": 224},
}

# --- Pretrained checkpoint directory ---
# Jitter checkpoints (jitter_{dataset}_{delay_tag}_trained.pt) are the
# shared "clean-trained SHD model" cache for all Phase 3 eval-only notebooks.
MODEL_DIR: str = "data"
CHECKPOINT_PREFIX: str = "jitter"  # filename stem in MODEL_DIR

# --- SLAYER neuron and simulation descriptors ---
SIM_PARAMS = {"Ts": 1, "tSample": 200}
LIF_PARAMS = {
    "type": "SRMALPHA",
    "theta": 10,
    "tauSr": 1,
    "tauRho": 0.1,
    "tauRef": 2,
    "scaleRef": 2,
    "scaleRho": 0.1,
}

# --- Data split ratios (test split only is consumed in eval-only mode) ---
TRAIN_RANGE = (0.0, 0.6)
VAL_RANGE = (0.6, 0.75)
TEST_RANGE = (0.75, 0.9)

# --- Architecture hyper-parameters (must match the trained models) ---
HIDDEN_UNITS: int = 128
NUM_CLASSES: int = 20
BATCH_SIZE: int = 128
SEED: int = 42
MAX_DELAY: int = 64

# --- Per-Spike Jitter sweep ---
SIGMA_VALUES: list[int] = [0, 1, 3, 5, 10, 17, 25]

# --- Evaluation ---
NUM_REPEATS: int = 3

# --- Derived names (default config; the grid loop derives its own per cell) ---
INPUT_DIM: int = DATASET_CONFIGS[DATASET_KEY]["input_dim"]
MAT_FILE: str = DATASET_CONFIGS[DATASET_KEY]["mat_file"]
DELAY_TAG: str = "delay" if USE_DELAY else "nodelay"
MODEL_PREFIX: str = f"jitter_{DATASET_KEY}_{DELAY_TAG}"

print(f"eval_all_variation = {eval_all_variation}")
print(f"Default dataset: SHD {DATASET_KEY} | Input dim: {INPUT_DIM}")
print(f"Default network mode: {'SGD-delay' if USE_DELAY else 'SGD (no delay)'}")
print(f"Checkpoint dir: {MODEL_DIR} (prefix={CHECKPOINT_PREFIX})")
print(f"Sweep (SIGMA_VALUES): {SIGMA_VALUES}")

## 3. Load SHD Dataset

Load the dense spike-train dataset from the local `.mat` file. Eval-only
mode only consumes the test split, but we keep the full loader signature
unchanged so the data utilities match `jitter_train.ipynb` exactly.

In [ ]:
def load_shd_data(mat_path: str, target_T: int = 200) -> tuple[np.ndarray, np.ndarray]:
    """Load SHD dataset from a .mat file and pad time dimension.

    Args:
        mat_path: Path to the .mat file containing 'X' and 'Y'.
        target_T: Target time dimension (pad with zeros if shorter).

    Returns:
        Tuple of (X, Y) where X has shape (N, neurons, target_T).
    """
    data = loadmat(mat_path)
    X = data["X"]
    Y = data["Y"].ravel()

    n_samples, n_neurons, T = X.shape
    if T < target_T:
        padded = np.zeros((n_samples, n_neurons, target_T), dtype=X.dtype)
        padded[:, :, :T] = X
        X = padded
        print(f"Padded time dimension from {T} to {target_T}")

    print(f"Loaded {mat_path}: X={X.shape}, Y={Y.shape}, classes={len(np.unique(Y))}")
    return X, Y


X_all, Y_all = load_shd_data(MAT_FILE, target_T=SIM_PARAMS["tSample"])

## 4. Per-Spike Jitter Utility

Identical to `jitter_train.ipynb` — see that notebook for the rationale
behind the per-spike Gaussian jitter choice. The functions are duplicated here so this
notebook is self-contained.

In [ ]:
def jitter_spike_train(
    spike_train: np.ndarray,
    sigma: float = 0.0,
    max_attempts: int = 50,
) -> np.ndarray:
    """Apply per-spike Gaussian jitter to a binary spike train.

    Each spike is shifted by a random offset ~ N(0, sigma). The new time
    is clipped to [0, T-1]. If the target bin is occupied, retry up to
    `max_attempts` times; if all fail, keep the spike at its original time.

    Args:
        spike_train: Binary array of shape (num_neurons, T).
        sigma: Standard deviation of the Gaussian jitter (in time steps / ms).
        max_attempts: Max retries per spike to find an empty bin.

    Returns:
        Jittered spike train with same shape.
    """
    if sigma <= 0:
        return spike_train.copy()

    num_neurons, T = spike_train.shape
    new_train = np.zeros_like(spike_train)

    for neuron_idx in range(num_neurons):
        spike_times = np.where(spike_train[neuron_idx] == 1)[0]
        if len(spike_times) == 0:
            continue

        for old_time in spike_times:
            inserted = False
            attempts = 0
            while not inserted and attempts < max_attempts:
                attempts += 1
                jittered_time = int(round(old_time + np.random.normal(0, sigma)))
                jittered_time = np.clip(jittered_time, 0, T - 1)

                if new_train[neuron_idx, jittered_time] == 0:
                    new_train[neuron_idx, jittered_time] = 1
                    inserted = True

            if not inserted:
                new_train[neuron_idx, old_time] = 1

    return new_train


def jitter_hidden_batch(
    hidden_spikes: torch.Tensor,
    sigma: float,
) -> torch.Tensor:
    """Apply per-spike jitter to a batch of hidden spike tensors.

    Converts from SLAYER's 5-D format (B, C, 1, 1, T) to numpy,
    jitters each sample, and converts back.

    Args:
        hidden_spikes: Spike tensor of shape (B, C, 1, 1, T).
        sigma: Jitter standard deviation in ms.

    Returns:
        Jittered spike tensor with same shape and device.
    """
    dev = hidden_spikes.device
    spikes_np = hidden_spikes.detach().cpu().numpy()
    B, C, H, W, T = spikes_np.shape

    for b in range(B):
        sample = spikes_np[b, :, 0, 0, :]  # (C, T)
        spikes_np[b, :, 0, 0, :] = jitter_spike_train(sample, sigma)

    return torch.from_numpy(spikes_np).to(dev)

## 5. Dataset and Data Splitting

A simple `Dataset` wrapper and a helper to split into train / validation /
test sets. Eval-only mode only uses the test loader, but the build returns
all three for parity with `jitter_train.ipynb`.

In [ ]:
class SpikeDataset(Dataset):
    """Wrap numpy spike trains and labels into a PyTorch Dataset."""

    def __init__(self, X: np.ndarray, Y: np.ndarray):
        self.X = X
        self.Y = Y

    def __len__(self) -> int:
        return len(self.Y)

    def __getitem__(self, idx: int):
        x = torch.tensor(self.X[idx], dtype=torch.float32)
        y = torch.tensor(self.Y[idx], dtype=torch.long)
        return x, y


def get_split_indices(
    split_range: tuple[float, float],
    total: int,
) -> np.ndarray:
    """Return index array for a given fractional range of the dataset."""
    start = int(total * split_range[0])
    end = int(total * split_range[1])
    return np.arange(start, end)


def build_dataloaders(
    X: np.ndarray,
    Y: np.ndarray,
    batch_size: int = 128,
    seed: int = 42,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Split data and build train/val/test DataLoaders."""
    N = len(Y)
    train_idx = get_split_indices(TRAIN_RANGE, N)
    val_idx = get_split_indices(VAL_RANGE, N)
    test_idx = get_split_indices(TEST_RANGE, N)

    np.random.seed(seed)
    np.random.shuffle(train_idx)

    train_ds = SpikeDataset(X[train_idx], Y[train_idx])
    val_ds = SpikeDataset(X[val_idx], Y[val_idx])
    test_ds = SpikeDataset(X[test_idx], Y[test_idx])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader

## 6. Network Architecture

Same 2-hidden-layer SLAYER SNN used in `jitter_train.ipynb`. Two forward
paths are kept so the model can be exercised in either clean or
perturbed mode:

- `forward(x)`: clean pass (matches how the jitter checkpoint was trained).
- `forward_with_hidden_perturbation(x, sigma)`: intercepts 1st hidden layer
  spikes, applies per-spike jitter, then continues to the rest of the network.
  Called only from eval loops inside `torch.no_grad()`.

`delay1` is applied at the start of `_second_hidden_and_output` so the
perturbation hook sees raw binary spikes (see Issue 2 in `phase1to4_fixes.md`).

In [ ]:
class JitterSHDNetwork(nn.Module):
    """2-hidden-layer SLAYER SNN with optional per-spike jitter at the 1st hidden layer."""

    def __init__(
        self,
        input_dim: int,
        hidden_units: int = 128,
        num_classes: int = 20,
        use_delay: bool = True,
        max_delay: int = 64,
    ):
        super().__init__()
        slayer = snn.layer(LIF_PARAMS, SIM_PARAMS)
        self.slayer = slayer
        self.use_delay = use_delay
        self.max_delay = max_delay

        self.fc1 = nn.utils.weight_norm(
            slayer.dense(input_dim, hidden_units), name="weight"
        )
        self.fc2 = nn.utils.weight_norm(
            slayer.dense(hidden_units, hidden_units), name="weight"
        )
        self.fc3 = nn.utils.weight_norm(
            slayer.dense(hidden_units, num_classes), name="weight"
        )

        if use_delay:
            self.delay1 = slayer.delay(hidden_units)
            self.delay2 = slayer.delay(hidden_units)

    def _prepare_input(self, x: torch.Tensor) -> torch.Tensor:
        """Ensure input is 5-D NCHWT on the correct device."""
        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x)
        if x.dim() == 3:
            x = x.unsqueeze(2).unsqueeze(3)
        return x.float().to(device)

    def _first_hidden(self, x: torch.Tensor) -> torch.Tensor:
        """Input -> PSP -> fc1 -> spike. Returns raw binary hidden1 spikes (pre-delay)."""
        return self.slayer.spike(self.fc1(self.slayer.psp(x)))

    def _second_hidden_and_output(self, hidden1: torch.Tensor) -> torch.Tensor:
        """(delay1) -> hidden1 -> PSP -> fc2 -> spike -> (delay2) -> PSP -> fc3 -> spike."""
        if self.use_delay:
            hidden1 = self.delay1(hidden1)
        x = self.slayer.spike(self.fc2(self.slayer.psp(hidden1)))
        if self.use_delay:
            x = self.delay2(x)
        x = self.slayer.spike(self.fc3(self.slayer.psp(x)))
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Clean forward pass — used when sigma is irrelevant."""
        x = self._prepare_input(x)
        hidden1 = self._first_hidden(x)
        return self._second_hidden_and_output(hidden1)

    def forward_with_hidden_perturbation(
        self,
        x: torch.Tensor,
        sigma: float = 0.0,
    ) -> torch.Tensor:
        """Forward pass with per-spike jitter at the 1st hidden layer."""
        x = self._prepare_input(x)
        hidden1 = self._first_hidden(x)

        if sigma > 0:
            hidden1 = jitter_hidden_batch(hidden1, sigma)

        return self._second_hidden_and_output(hidden1)

    def get_delays(self) -> dict[str, np.ndarray]:
        """Return current delay values as a dict."""
        delays = {}
        if self.use_delay:
            delays["delay1"] = self.delay1.delay.data.cpu().numpy()
            delays["delay2"] = self.delay2.delay.data.cpu().numpy()
        return delays

## 7. Load Pretrained Models

Replaces the training loop from `jitter_train.ipynb`. The helper
`load_pretrained_model` reads a `state_dict` from disk and returns a
ready-to-eval network. The grid loop (cell 20) calls it once per
`(dataset_key, use_delay)` configuration.

In [ ]:
def load_pretrained_model(
    model_path: str,
    input_dim: int,
    hidden_units: int = HIDDEN_UNITS,
    num_classes: int = NUM_CLASSES,
    use_delay: bool = False,
    max_delay: int = MAX_DELAY,
) -> JitterSHDNetwork:
    """Load a pretrained JitterSHDNetwork from a checkpoint file.

    Args:
        model_path: Path to the saved .pt state dict.
        input_dim: Number of input neurons.
        hidden_units: Hidden layer size.
        num_classes: Number of output classes.
        use_delay: Whether the saved model used learnable delays.
        max_delay: Maximum delay (only relevant when ``use_delay=True``).

    Returns:
        The loaded network in eval mode on the active device.
    """
    net = JitterSHDNetwork(
        input_dim, hidden_units, num_classes, use_delay, max_delay
    ).to(device)

    state_dict = torch.load(model_path, map_location=device, weights_only=True)
    net.load_state_dict(state_dict)
    net.eval()

    print(f"  Loaded model from {model_path}")
    print(f"    Architecture: {input_dim} -> {hidden_units} -> {hidden_units} -> {num_classes}")
    print(f"    Delay: {'Yes' if use_delay else 'No'}")
    return net

## 8. Testing with Hidden-Layer Jitter

Evaluate the loaded model on the test set by sweeping sigma values.
Each call goes through `forward_with_hidden_perturbation` inside
`torch.no_grad()`. Repeated evaluations with different random seeds
provide error bars.

In [ ]:
def test_with_jitter(
    net: JitterSHDNetwork,
    test_loader: DataLoader,
    sigma: float = 0.0,
) -> float:
    """Evaluate accuracy with hidden-layer per-spike jitter at level sigma."""
    net.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.unsqueeze(2).unsqueeze(3).float().to(device)
            y_batch = y_batch.to(device)

            outputs = net.forward_with_hidden_perturbation(x_batch, sigma=sigma)
            predicted = snn.predict.getClass(outputs)

            total += y_batch.size(0)
            correct += (predicted.cpu() == y_batch.cpu()).sum().item()

    return correct / total


def test_with_repeats(
    net: JitterSHDNetwork,
    test_loader: DataLoader,
    sigma: float,
    num_repeats: int = NUM_REPEATS,
) -> dict:
    """Evaluate with per-spike jitter multiple times for error bars."""
    accuracies = []
    for repeat in range(num_repeats):
        np.random.seed(SEED + repeat)
        acc = test_with_jitter(net, test_loader, sigma=sigma)
        accuracies.append(acc)

    return {
        "mean": float(np.mean(accuracies)),
        "std": float(np.std(accuracies)),
        "values": accuracies,
    }

## 9. Visualisation Utilities

Plot accuracy vs. perturbation level across the trained configurations.
Eval-only mode has no training-curve plot since there is no training log.

In [ ]:
def plot_jitter_sweep_curve(
    sweep_results: dict[int, dict],
    dataset_key: str | None = None,
    delay_tag: str | None = None,
    use_delay: bool | None = None,
    model_prefix: str | None = None,
) -> None:
    """Plot accuracy vs hidden per-spike jitter for one configuration."""
    dk = dataset_key if dataset_key is not None else DATASET_KEY
    dt = delay_tag if delay_tag is not None else DELAY_TAG
    ud = use_delay if use_delay is not None else USE_DELAY
    mp = model_prefix if model_prefix is not None else MODEL_PREFIX

    x_vals = sorted(sweep_results.keys())
    means = [sweep_results[v]["mean"] for v in x_vals]
    stds = [sweep_results[v]["std"] for v in x_vals]

    color = "tab:orange" if ud else "tab:blue"
    label = f"SGD-delay ({dk})" if ud else f"SGD ({dk})"

    plt.figure(figsize=(8, 5))
    plt.errorbar(
        x_vals, means, yerr=stds, fmt="o-",
        capsize=5, capthick=2, color=color, label=label,
    )
    plt.xlabel("Hidden Jitter Sigma (ms)")
    plt.ylabel("Test Accuracy")
    plt.title(
        f"Exp 3A: SHD {dk} — Accuracy vs Hidden Per-Spike Jitter ({dt}, eval-only)"
    )
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()

    for x_val, mean, std in zip(x_vals, means, stds):
        plt.annotate(
            f"{mean:.3f}",
            (x_val, mean),
            textcoords="offset points",
            xytext=(0, 12),
            ha="center",
            fontsize=9,
        )

    plt.tight_layout()
    fig_path = f"log/{mp}_evalOnly_jitter_sweep.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Figure saved to {fig_path}")

## 10. Run: Load Pretrained Models, then Sweep Jitter at Eval Time

When `eval_all_variation = False` (default), load a single pretrained
model for the `(DATASET_KEY, USE_DELAY)` configuration selected at the
top and run the sigma sweep against it.

When `eval_all_variation = True`, the loop iterates over **all six**
combinations of dataset ∈ {whole, part, norm} × use_delay ∈ {False, True},
loading each `jitter_{dataset}_{delay_tag}_trained.pt` checkpoint and
running the sweep on it. Results are accumulated in
`all_runs[(dataset_key, use_delay)]` so the downstream plot / save /
analysis cells iterate over every configuration.

In [ ]:
os.makedirs("log", exist_ok=True)

# Build the configuration grid. When eval_all_variation is True we sweep all
# (dataset, use_delay) combinations; otherwise we run the single default
# config picked at the top of the notebook.
if eval_all_variation:
    config_grid: list[tuple[str, bool]] = [
        (dk, ud) for dk in DATASET_CONFIGS for ud in (False, True)
    ]
else:
    config_grid = [(DATASET_KEY, USE_DELAY)]

print(f"Running {len(config_grid)} configuration(s): {config_grid}")

all_runs: dict[tuple[str, bool], dict] = {}

for dataset_key, use_delay in config_grid:
    input_dim = DATASET_CONFIGS[dataset_key]["input_dim"]
    mat_file = DATASET_CONFIGS[dataset_key]["mat_file"]
    delay_tag = "delay" if use_delay else "nodelay"
    model_prefix = f"jitter_{dataset_key}_{delay_tag}"
    ckpt_path = os.path.join(
        MODEL_DIR, f"{CHECKPOINT_PREFIX}_{dataset_key}_{delay_tag}_trained.pt"
    )

    print(f"\n{'#'*70}")
    print(f"#  Configuration: dataset={dataset_key} | {delay_tag}")
    print(f"{'#'*70}")

    # Load test data for this configuration
    X_cfg, Y_cfg = load_shd_data(mat_file, target_T=SIM_PARAMS["tSample"])
    _, _, test_loader = build_dataloaders(
        X_cfg, Y_cfg, batch_size=BATCH_SIZE, seed=SEED
    )

    # Load the pretrained model
    print(f"\n  --- Loading pretrained model ---")
    cfg_net = load_pretrained_model(
        model_path=ckpt_path,
        input_dim=input_dim,
        use_delay=use_delay,
    )

    # Evaluate across the sweep
    print(f"\n  --- Hidden-jitter sweep at evaluation ---")
    cfg_test_results: dict[int, dict] = {}
    for sigma in SIGMA_VALUES:
        test_result = test_with_repeats(cfg_net, test_loader, sigma=sigma)
        cfg_test_results[sigma] = test_result
        print(
            f"    sigma={sigma:3d} ms | "
        f"accuracy = {test_result['mean']:.4f} +/- {test_result['std']:.4f}"
        )

    all_runs[(dataset_key, use_delay)] = {
        "net": cfg_net,
        "all_test_results": cfg_test_results,
        "dataset_key": dataset_key,
        "use_delay": use_delay,
        "delay_tag": delay_tag,
        "model_prefix": model_prefix,
        "ckpt_path": ckpt_path,
    }

# Expose the last run's results as module-level names for any cell that
# still references them directly.
_last = list(all_runs.values())[-1]
net = _last["net"]
all_test_results = _last["all_test_results"]

## 11. Plot Jitter Sweep Results

The main result: test accuracy vs. hidden-layer sigma evaluated on
the pretrained model(s).

In [ ]:
for (_dk, _ud), _run in all_runs.items():
    print(f"\n--- Sweep curve: dataset={_dk}, {_run['delay_tag']} ---")
    plot_jitter_sweep_curve(
        _run["all_test_results"],
        dataset_key=_dk,
        delay_tag=_run["delay_tag"],
        use_delay=_ud,
        model_prefix=_run["model_prefix"],
    )

## 12. Save Results

Save the sweep results to JSON. Filenames carry an `_evalOnly` tag so
they do not collide with the train-mode outputs.

In [ ]:
for (_dk, _ud), _run in all_runs.items():
    _mp = _run["model_prefix"]

    sweep_serialisable = {
        str(sigma): {
            "mean": data["mean"],
            "std": data["std"],
            "values": [float(v) for v in data["values"]],
        }
        for sigma, data in _run["all_test_results"].items()
    }
    results_path = f"log/{_mp}_evalOnly_jitter_sweep_results.json"
    with open(results_path, "w") as fp:
        json.dump(sweep_serialisable, fp, indent=2)
    print(f"Sweep results saved to {results_path}")

## 13. Model Analysis

Print delay statistics and weight statistics for each loaded model.

In [ ]:
for (_dk, _ud), _run in all_runs.items():
    print(f"\n{'='*60}")
    print(f"  Model Analysis: dataset={_dk}, {_run['delay_tag']}")
    print(f"  Checkpoint: {_run['ckpt_path']}")
    print(f"{'='*60}")
    _cfg_net = _run["net"]

    delays = _cfg_net.get_delays()
    if delays:
        for delay_name, delay_values in delays.items():
            if len(delay_values) > 0:
                print(
                    f"  {delay_name}: "
                    f"mean={np.mean(delay_values):.2f}, "
                    f"std={np.std(delay_values):.2f}, "
                    f"min={np.min(delay_values):.2f}, "
                    f"max={np.max(delay_values):.2f}"
                )
    else:
        print("  No delays (SGD mode)")

    for name, param in _cfg_net.named_parameters():
        if "weight" in name:
            w = param.data
            print(
                f"  {name}: mean={w.mean().item():.4f}, "
                f"std={w.std().item():.4f}, "
                f"shape={list(w.shape)}"
            )